# Seurat Label Transfer


**Pinned Environment:** [`envs/R_integration.yaml`](../../envs/R_integration.yaml)

- Reference mapping guide: https://www.10xgenomics.com/analysis-guides/xenium-cell-type-annotation
- This notebook uses BP cells, which can be installed in R with remotes::install_github("bnprks/BPCells/r")

In [ ]:
packageVersion("Seurat")

In [ ]:
library(Seurat)
library(BPCells)
library(SeuratObject)
library(SeuratDisk)
library(tidyverse)
library(jsonlite)
options(future.globals.maxSize = 1e9)

In [ ]:
#library(SingleCellExperiment)

In [ ]:
packageVersion("Seurat")

In [ ]:
# For plotting
library(ggplot2)
library(ggpmisc)
library(scales)
library(cowplot)
library(gridExtra)
library(viridis)
library(hrbrthemes)

## File info

In [ ]:
source("../../config/paths.R")

# Folder where the .rds files were saved
input_dir <- file.path(BASE_DIR, "data/rds/seurat")

# Full paths to the Xenium and reference RDS files
xenium_rds_file  <- file.path(input_dir, "adata-xenium.rds")
refdata_rds_file <- file.path(input_dir, "refdata-seq.rds")

# Output folder for downstream label transfer or plots
output_dir <- file.path(BASE_DIR, "data/rds/seurat")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

## Load reference reference


In [ ]:
ref <- readRDS(refdata_rds_file)

In [ ]:
group_counts_df <- as.data.frame(table(ref$cell_label))
colnames(group_counts_df) <- c("group", "count")

print("Annotation counts:")
print(group_counts_df)

## Load Xenium 

In [ ]:
xenium.obj <- readRDS(xenium_rds_file)
DefaultAssay(xenium.obj) <- "Xenium"

In [ ]:
xenium.obj

## Process xenium

### BP cells

In [ ]:
# Path to store on-disk BPCells matrix
counts_dir <- file.path(input_dir, "xenium_counts_bpcells")

if (dir.exists(counts_dir)) {
  unlink(counts_dir, recursive = TRUE)
}

# Write the counts matrix to disk (BPCells format)
write_matrix_dir(
  mat = xenium.obj[["Xenium"]]$counts,
  dir = counts_dir
)

# Reload matrix in on-disk format and assign back into Seurat object
counts.mat <- open_matrix_dir(dir = counts_dir)
xenium.obj[["Xenium"]]$counts <- counts.mat

rm(counts.mat)

### Xenium QC

In [ ]:
print(median(xenium.obj@meta.data$nCount_Xenium))
print(median(xenium.obj@meta.data$nFeature_Xenium))

# Add log1p_nCount_RNA, log1p_nFeatures
xenium.obj@meta.data$nCount_Xenium_log <- log1p(xenium.obj@meta.data$nCount_Xenium)
xenium.obj@meta.data$nFeature_Xenium_log <- log1p(xenium.obj@meta.data$nFeature_Xenium)

# remove cells with < 50 counts (aggressive filter)
xenium.obj <- subset(xenium.obj, subset = nCount_Xenium > 50)

# Violinplots of the transcript and feature counts per cell
VlnPlot(xenium.obj, features = c("nCount_Xenium_log", "nFeature_Xenium_log"), ncol = 2, pt.size = 0, group.by = "orig.ident")

In [ ]:
# This function takes in the Xenium and single cell reference Seurat object and returns the per-gene expression means

get_gex_means <- function(xenium_obj, ref_obj){

    xen_means <- data.frame(
        mean_counts = rowMeans(xenium_obj[["Xenium"]]$counts),
        gene = rownames(xenium_obj[["Xenium"]]$counts)
    ) %>%
        arrange(desc(mean_counts)) %>%
        mutate(Rank = 1:n())

    ref_means <- data.frame(
        mean_counts = rowMeans(ref_obj[["RNA"]]$counts),
        gene = rownames(ref_obj[["RNA"]]$counts)
    ) %>%
        arrange(desc(mean_counts)) %>%
        mutate(Rank = 1:n())

    # Merge mean counts per cell,
    merged_means <- merge(xen_means, ref_means, by.x = "gene", by.y = "gene", all.x = TRUE)

    return(merged_means)
}


In [ ]:
merged_means <- get_gex_means(xenium.obj, ref)

In [ ]:
ggplot(merged_means, aes(x=mean_counts.y, y=mean_counts.x)) +
    geom_point(size=0.5) +
    scale_colour_manual(values=c("darkcyan", "coral")) +
    stat_poly_eq() +
    scale_x_log10() +
    scale_y_log10() +
    xlab("Ref Mean Expression") + ylab("Xenium Mean Expression") +
    ggtitle("correlation") +
    theme_classic() +
    theme(axis.text = element_text(color="black", size=10),
          axis.title = element_text(size=12)) +
    geom_abline(slope = 1, intercept = 0) 
    # tune::coord_obs_pred()

### Normalization, umap on reference

In [ ]:
DefaultAssay(ref) <- "RNA"
ref <- NormalizeData(ref) %>%
                 FindVariableFeatures() %>%
                 ScaleData() %>%
                 RunPCA() %>%
                 RunUMAP(dims=1:15) %>%
                 FindNeighbors(dims=1:15) %>%
                 FindClusters(resolution=0.5)

#### Make named colors for plotting labels consistently 

In [ ]:
n_colors <- length(unique(ref$cell_label))
colors_polychrome <- Seurat::DiscretePalette(n = n_colors, palette = "polychrome")
names(colors_polychrome) <-unique(ref$cell_label)

In [ ]:
DimPlot(ref, reduction = "umap", cols = "polychrome", group.by = "RNA_snn_res.0.5")

In [ ]:
DimPlot(ref, reduction = "umap", 
              cols = colors_polychrome, 
        label=T,
              group.by = "cell_label") +
  theme(
    legend.position = "bottom",       
    legend.text = element_text(size = 10),
    legend.key.size = unit(0.5, "cm")
  )

### Normalize, umap on Xenium 

In [ ]:
DefaultAssay(xenium.obj) <- "Xenium"
xenium.obj <- NormalizeData(xenium.obj)
xenium.obj <- FindVariableFeatures(xenium.obj)

DefaultAssay(xenium.obj) <- "Xenium"

# process xenium 
xenium.obj <- FindVariableFeatures(xenium.obj) %>%
              ScaleData() %>%
              RunPCA(npcs = 20) %>%
              RunUMAP(dims = 1:16, return.model=TRUE) %>%
              FindNeighbors(reduction = "pca", dims = 1:16) %>%
              FindClusters(resolution = 0.6)

# Plot UMAP
DimPlot(xenium.obj, group.by = "Xenium_snn_res.0.6", 
        cols = "polychrome", label=TRUE, label.size = 4)

markers <- c('leiden_scVI_1')
FeaturePlot(xenium.obj, features = markers)

In [ ]:
xenium.obj

## Run FindAnchors for Label Transfer

In [ ]:
# Run on gene intersection
common_genes <- intersect(rownames(xenium.obj), rownames(ref))
print(length(common_genes))

print('subset ref to common genes and process ref')
ref_subset <- CreateSeuratObject(counts = ref[["RNA"]]$counts[common_genes,],
                                 meta = ref@meta.data) %>%
                                NormalizeData() %>%
                                FindVariableFeatures() %>%
                                ScaleData() %>%
                                RunPCA() %>%
                                RunUMAP(dims=1:15) 

# Visualize
DimPlot(ref_subset, reduction = "umap", 
              # cols = colors_polychrome, 
        cols = "polychrome", 
        label=T,
              group.by = "cell_label") +
  theme(
    legend.position = "bottom",       
    legend.text = element_text(size = 10),
    legend.key.size = unit(0.5, "cm")
  )

# Find anchors
print('Find Anchors')
ref[["RNA"]]$counts <- as(object = ref[["RNA"]]$counts, Class = "dgCMatrix")

xenium.obj[["Xenium"]]$counts <- as(object = xenium.obj[["Xenium"]]$counts, Class = "dgCMatrix")

anchors_from_ref <- FindTransferAnchors(reference = ref_subset,
                                         query = xenium.obj,
                                         query.assay = "Xenium",
                                         features = common_genes,
                                         dims = 1:20,
                                         reference.reduction = "pca")

# Label transfer
print('label transfer')
label_transfer <- TransferData(anchorset = anchors_from_ref,
                               refdata = ref_subset$cell_label,
                               dims = 1:20)

# add predicted id
xenium.obj <- AddMetaData(object = xenium.obj, metadata = label_transfer, col.name = 'predicted.id')

# visualize
DimPlot(xenium.obj, group.by = c("predicted.id"), 
        cols = "polychrome", 
        label=T, label.size = 3)+
  theme(
    legend.position = "bottom",       
    legend.text = element_text(size = 10),
    legend.key.size = unit(0.5, "cm")
  )

In [ ]:
# Export RDS
saveRDS(xenium.obj, file.path(output_dir, 'xenium_processed_labeltransfer.rds'))

## Xenium and Sequencing Co-Embedding

In [ ]:
# Standard PCA/UMAP
standard_dim_reduction <- function(obj, seed = 1234){

    DefaultAssay(obj) <- "RNA"
    set.seed(seed)

    obj <- NormalizeData(obj)
    obj <- FindVariableFeatures(obj)
    obj <- ScaleData(obj)
    obj <- RunPCA(obj)

    obj <- FindNeighbors(obj, dims = 1:10, reduction = "pca")
    obj <- FindClusters(obj, resolution = 1, cluster.name = "unintegrated_clusters")

    obj <- RunUMAP(obj, dims = 1:10, reduction = "pca",
                   reduction.name = "umap.unintegrated")

    return(obj)
}

In [ ]:
# CCA integration
integrate_objects <- function(comb_obj,
                              integration_method = CCAIntegration,
                              reduction_name = "integrated.cca",
                              seed = 1234){

    set.seed(seed)

    comb_obj <- IntegrateLayers(
        object = comb_obj,
        method = integration_method,
        orig.reduction = "pca",
        new.reduction = reduction_name,
        verbose = FALSE
    )

    # After integration, rejoin layers
    comb_obj[["RNA"]] <- JoinLayers(comb_obj[["RNA"]])

    comb_obj <- FindNeighbors(comb_obj, reduction = reduction_name, dims = 1:20)
    comb_obj <- FindClusters(comb_obj, resolution = 1)

    comb_obj <- RunUMAP(comb_obj, dims = 1:20,
                        reduction = reduction_name,
                        seed.use = seed)

    return(comb_obj)
}

In [ ]:
# Prepare Xenium + Reference
xenium_full <- xenium.obj
DefaultAssay(xenium_full) <- "Xenium"

xenium_full_layer <- DietSeurat(
    object = xenium_full,
    assays = "Xenium",
    layers = "counts"
)
xenium_full_layer <- RenameAssays(xenium_full_layer, Xenium = "RNA")

ref_layer <- DietSeurat(
    object = ref_subset,
    assays = "RNA",
    layers = "counts"
)

# Tag modality
ref_layer$modality <- "ref"
xenium_full_layer$modality <- "xen"

# Merge full datasets
comb <- merge(ref_layer, y = xenium_full_layer)
cat("Full combined dataset:", paste(dim(comb), collapse = " x "), "\n")

In [ ]:
# Run PCA / CCA / UMAP

comb <- standard_dim_reduction(comb)
comb_cca <- integrate_objects(comb, reduction_name = "integrated.cca", seed = 1)

# Save RDS
saveRDS(comb_cca, file.path(output_dir, "xenium_ref_comb_full.rds"))

In [ ]:
DimPlot(comb, reduction = "umap.unintegrated", group.by = "modality")
DimPlot(comb_cca, reduction = "umap", group.by = "modality")

DimPlot(
    subset(comb_cca, modality == "xen"),
    reduction = "umap",
    group.by = "predicted.id",
    label = TRUE,
    label.size = 3,
    cols = colors_polychrome
) +
  theme(
    legend.position = "bottom",
    legend.text = element_text(size = 10),
    legend.key.size = unit(0.5, "cm")
  )

DimPlot(
    subset(comb_cca, modality == "ref"),
    reduction = "umap",
    group.by = "cell_label",
    label = TRUE,
    label.size = 3,
    cols = colors_polychrome
) +
  theme(
    legend.position = "bottom",
    legend.text = element_text(size = 10),
    legend.key.size = unit(0.5, "cm")
  )

In [ ]:
write.csv(label_transfer, file = file.path(output_dir, "label_transfer_xenium.csv"), row.names = TRUE)